# Unit 13 solutions: race day

These reference solutions check section 0's lower bound against the line's curvature, solve `monza-like`'s line again, replay the re-tune's record and catch its floating-point tie, apply the margin on every knob at its edge on a held-out car, choose between MPC and pursuit, price a crash, look at the timing policy, and read the race-day file as a reviewer would; the Challenge asks whether a faster setting is a finding or a fit.
Each exercise's statement is repeated, followed by a prediction made before running anything, then the solution.
Every race is seeded and uses `MPC_BUDGET_MS = None` except Exercise 7's one timing-on race, so every race number below is reproducible, and executable asserts check each answer; step times depend on the machine and are printed, never asserted.
Every number is a number about racekit's **carla-like car** and its perturbed family: a model, not CARLA's car.
**No race here touches the test cars 72 to 95** (the setup's `race()` refuses them), and none touches `test-circuit-seed2027` or `test-circuit-seed2028`.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_11, UNIT_13 = AGENTS / "unit-07.py", AGENTS / "unit-11.py", AGENTS / "unit-13.py"
workdir = Path(mkdtemp(prefix="roar-unit-13-sol-"))
TRACK = "monza-like"
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
CAP = 900.0  # s: a monza-like race is cut here
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
MONZA_MU_G = (5.38, 8.96, 9.12, 9.14, 9.70, 7.97)  # m/s^2: Unit 8's six grip probes on monza-like

# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x

NAME_CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"


def constant_name(line):
    # The UPPER_CASE name of a module-constant line ("NAME = value  # comment"), or None (Unit 12's rule).
    cuts = [i for i in range(len(line) - 2) if line[i:i + 3] == " = "]
    if cuts and line[0] in NAME_CHARACTERS[:26] and all(c in NAME_CHARACTERS for c in line[:cuts[0]]):
        return line[:cuts[0]]
    return None


def module_constants(path):
    # name -> the value's text (after " = ", without a "  # ..." comment), for every module-constant line.
    # A value written over several lines (LINE_CONTROL) is only its first line here: "(".
    constants = {}
    with open(path) as handle:
        for line in handle:
            name = constant_name(line)
            if name is not None:
                text = line[len(name) + 3:-1]
                comment = [i for i in range(len(text) - 2) if text[i:i + 3] == "  #"]
                constants[name] = text[:comment[0]] if comment else text
    return constants


def file_parts(path):
    # (the comment lines above the docstring, the rest of the file without its constant lines and without
    # the line's values): the header may change, the rest must not.
    with open(path) as handle:
        lines = [line for line in handle]
    start = [i for i, line in enumerate(lines) if line[:3] == '"""'][0]
    body, in_line = [], False
    for line in lines[start:]:
        if line[:15] == "LINE_CONTROL = ":
            in_line = True
        elif in_line:
            in_line = line != ")\n"
        elif constant_name(line) is None:
            body.append(line)
    return lines[:start], body


track = load_track(TRACK)
with shim_installed():
    Unit11, name11 = load_submission(UNIT_11)
    Unit13, name13 = load_submission(UNIT_13)
eleven, thirteen = sys.modules[name11], sys.modules[name13]  # the two files' constants and functions
agent13 = Unit13(track.waypoints, None)  # unit-13.py's agent on monza-like (no race): its line, curvature, profile
LINE = np.array(agent13.offsets)  # m: unit-13.py's racing line, as offsets from the centreline
written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def race(source=UNIT_13, seed=0, k=None, laps=3, **constants):
    # One race on monza-like of `source` with `constants` changed and the timing off, on the carla-like car with
    # its noise (car k of Unit 11's perturbed family if k is given: pass seed=k % 5).
    assert k is None or k < 72, "cars 72-95 are the test cars, and k >= 1000 are sealed: not here"
    path = variant(source, **constants, **NO_TIMING)
    return run_race(path, TRACK, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=CAP)


def error_trace(result, offsets=LINE):
    # Unit 9: the time and the cross-track error from the race's own line over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return telemetry["t"][driving], telemetry["offset"][driving] - target


def line_error(result, offsets=LINE):
    return float(np.max(np.abs(error_trace(result, offsets)[1])))


def clean(result, offsets=LINE):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def same_race(a, b):
    # The same race: every telemetry row equal, not only the time.
    return len(a.telemetry) == len(b.telemetry) and all(
        np.sum(a.telemetry[name] != b.telemetry[name]) == 0 for name in a.telemetry.dtype.names)

## Exercise 1 - Section 0's lower bound (`cornering-speed-limit`, `speed-profile`)

Unit 8's probe of section 0 on `monza-like` never slid, so `MU_G[0] = 5.38` is a **lower bound** on that section's grip.
Find out, without a race, whether that can cost any time.
`agent13` (the setup) is `unit-13.py`'s agent built on `monza-like`: `agent13.curvature` is the signed curvature at every point of its racing line (1/m), `agent13.section` the grip section of every point, and `agent13.profile` the speed profile it plans (m/s, Unit 7).
The planner's corner cap in section s is min(`TARGET_SPEED`, √(`MU_FRACTION` · `MU_G[s]` / |κ|)) (Units 7 and 8).
This exercise uses the measured grip; it identifies nothing.

Define:

- `kappa_max`: a dict from each section 0 to 5 to the largest |κ| on the line in that section (1/m);
- `v_max`: a dict from each section to √(`MU_G[s]` / `kappa_max[s]`), the fastest corner speed at the full measured grip (m/s);
- `planner_cap`: a dict from each section to the planner's cap at `kappa_max[s]` (m/s, at most `TARGET_SPEED`);
- `limited`: the sorted list of sections whose `planner_cap` is below `TARGET_SPEED` (where the grip sets the speed);
- `grip_needed`: the grip μ·g (m/s²) below which section 0's cap would drop below `TARGET_SPEED`;
- `high_mu_agent`: the agent of `unit-13.py` with `MU_G[0] = 12.0` (and the other five sections unchanged) built on `monza-like` (`load_submission(variant(UNIT_13, MU_G=...))` inside `shim_installed()`, then the class with `track.waypoints` and `None`), and `profile_same`: `True` when its `profile` equals `agent13.profile`;
- `fig`, `ax`: `agent13.profile` against the waypoint index, with `planner_cap`'s value for each section drawn as a horizontal segment over that section's waypoints, axis labels and a title;
- `why_free`: two sentences: why does section 0's lower bound cost nothing on this line, and when would a lower bound on grip cost time?

**Prediction.** The lesson found section 0's sharpest point on the line at about 2e-5 1/m (a radius of about 45 km), so its corner speed at the full grip is hundreds of m/s and its cap is the top speed, 45 m/s.
The other five sections have real corners and should all be limited.
Section 0's cap drops below 45 m/s only below about 0.07 m/s² of grip, so changing `MU_G[0]` from 5.38 to 12.0 should leave the planned profile exactly the same.

In [ ]:
curvature = np.abs(np.array(agent13.curvature))  # 1/m, on the racing line
section = np.array(agent13.section)
fraction, top = thirteen.MU_FRACTION, thirteen.TARGET_SPEED
kappa_max, v_max, planner_cap = {}, {}, {}
for s in range(6):
    kappa_max[s] = float(np.max(curvature[section == s]))
    v_max[s] = math.sqrt(MONZA_MU_G[s] / kappa_max[s])
    planner_cap[s] = min(top, math.sqrt(fraction * MONZA_MU_G[s] / kappa_max[s]))
limited = sorted([s for s in range(6) if planner_cap[s] < top])
grip_needed = top ** 2 * kappa_max[0] / fraction  # m/s^2: sqrt(fraction * mu_g / kappa) = top

with shim_installed():
    HighMu, _ = load_submission(variant(UNIT_13, MU_G=(12.0,) + MONZA_MU_G[1:]))
high_mu_agent = HighMu(track.waypoints, None)
profile_same = len(high_mu_agent.profile) == len(agent13.profile) and int(
    np.sum(np.array(high_mu_agent.profile) != np.array(agent13.profile))) == 0

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(range(len(agent13.profile)), agent13.profile, linewidth=0.7, label="agent13.profile (planned speed)")
index = np.arange(len(section))
for s in range(6):
    points = index[section == s]
    ax.plot([int(np.min(points)), int(np.max(points))], [planner_cap[s], planner_cap[s]], color="black",
            linestyle="--", linewidth=1.2, label="the planner's cap per section" if s == 0 else None)
ax.set_xlabel("waypoint index")
ax.set_ylabel("speed (m/s)")
ax.set_title("unit-13.py on monza-like: the planned speed and each section's corner cap")
ax.legend()
plt.show()

print("section  largest |kappa| (1/m)  radius (m)  v_max at mu*g (m/s)  the planner's cap (m/s)")
for s in range(6):
    print(f"{s:7d}  {kappa_max[s]:21.2e}  {1 / kappa_max[s]:10.0f}  {v_max[s]:19.1f}  {planner_cap[s]:23.2f}")
print("grip-limited sections:", limited)
print(f"section 0's cap drops below {top:.0f} m/s only below {grip_needed:.4f} m/s^2 (measured: at least {MONZA_MU_G[0]})")
print("MU_G[0] = 12.0 plans the same profile:", profile_same)
why_free = (
    f"On this line section 0 is a straight: its largest curvature is {kappa_max[0]:.1e} 1/m (a radius of about "
    f"{1 / kappa_max[0] / 1000:.0f} km), so even the planner's {fraction} of the grip caps it at "
    f"{math.sqrt(fraction * MONZA_MU_G[0] / kappa_max[0]):.0f} m/s, far above TARGET_SPEED {top:.0f} m/s; any grip "
    f"above {grip_needed:.3f} m/s^2 gives the same cap, so 5.38 and 12.0 plan the very same profile. "
    "A lower bound costs time only in a section where the corner cap is below the top speed (here sections 1 to 5, "
    f"capped at {min(planner_cap[s] for s in limited):.1f} to {max(planner_cap[s] for s in limited):.1f} m/s): "
    "there the planner would brake for grip the car really has, and on another line or track section 0 itself "
    "might hold a corner."
)
print(why_free)

assert sorted(kappa_max) == [0, 1, 2, 3, 4, 5] and kappa_max[0] < 3e-5
assert all(kappa_max[s] > 5e-3 for s in range(1, 6))
assert planner_cap[0] == top and v_max[0] > 400.0
assert limited == [1, 2, 3, 4, 5]
assert abs(grip_needed - 0.0705) <= 1e-4
assert profile_same

## Exercise 2 - The line, solved again (`racing-line`, `line-retune`)

A racing line belongs to one track: `unit-11.py` carries `test-circuit`'s and `unit-13.py` carries `monza-like`'s.
Solve `monza-like`'s line with Unit 9's solver (the setup's `solve_line`, about 2 s) and check it against the file.

Define:

- `monza_control`: `solve_line(track)`, and `monza_offsets`: the offset at every waypoint, `spline_basis(len(track)) @ monza_control` (m);
- `room`: `track.half_width - 2.5`, the room the solver allows on each side of the centreline (m), and `inside_width`: `True` when every |offset| is at most its `room` plus 1e-9;
- `pinned_ends`: `True` when every |offset| over the first 30 and the last 30 waypoints is at most 1e-9 (the start and finish stay on the centreline);
- `largest_offset`: the largest |offset| (m);
- `shipped_control`: `thirteen.LINE_CONTROL` as a numpy array, and `same_line`: `np.allclose(monza_control, shipped_control, rtol=0.0, atol=1e-6)` (the line solved here against the 285 values the file ships, each written with `repr`); `max_difference`: the largest |difference| (m);
- `values_per_track`: a dict `{"test-circuit": ..., "monza-like": ...}` with the number of control values in `eleven.LINE_CONTROL` and `thirteen.LINE_CONTROL`;
- `fig`, `ax`: the centreline (`track.x`, `track.y`) and the racing line (each waypoint moved `monza_offsets` along its left normal, `normals(track.x, track.y)`), equal axes, a legend and a title;
- `why_per_track`: two sentences: why can `unit-11.py`'s line not race `monza-like`, and what has to be done again when the race's track changes?

**Prediction.** The lesson solved this line and found it equal to the file's within 1e-6 m: 285 control values for 2,857 waypoints.
It should stay inside the room everywhere and be exactly 0 over the first and last 30 waypoints (the pinned start and finish).
`unit-11.py` ships 133 values for `test-circuit`'s 1,331 waypoints, and the agent refuses a line whose length does not match its track.

In [ ]:
monza_control = solve_line(track)
monza_offsets = spline_basis(len(track)) @ monza_control
room = track.half_width - 2.5
inside_width = bool(np.max(np.abs(monza_offsets) - room) <= 1e-9)
pinned_ends = bool(np.max(np.abs(np.concatenate([monza_offsets[:30], monza_offsets[-30:]]))) <= 1e-9)
largest_offset = float(np.max(np.abs(monza_offsets)))
shipped_control = np.array(thirteen.LINE_CONTROL)
same_line = np.allclose(monza_control, shipped_control, rtol=0.0, atol=1e-6)
max_difference = float(np.max(np.abs(monza_control - shipped_control)))
values_per_track = {"test-circuit": len(eleven.LINE_CONTROL), "monza-like": len(thirteen.LINE_CONTROL)}

nx, ny = normals(track.x, track.y)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(track.x, track.y, color="grey", linewidth=0.8, label="centreline")
ax.plot(track.x + monza_offsets * nx, track.y + monza_offsets * ny, linewidth=0.8, label="racing line (solved here)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("monza-like: Unit 9's smoothest line")
ax.legend()
plt.show()

print(f"{len(track)} waypoints, {len(monza_control)} control values; inside the room {inside_width}, "
      f"ends pinned {pinned_ends}, largest |offset| {largest_offset:.2f} m")
print(f"against the file's LINE_CONTROL: allclose {same_line}, largest |difference| {max_difference:.1e} m")
print("control values per track:", values_per_track)
why_per_track = (
    f"A line is a list of offsets along one track's waypoints: unit-11.py's {values_per_track['test-circuit']} values "
    f"were solved for test-circuit's 1,331 waypoints, and monza-like has {len(track):,} waypoints and needs "
    f"{values_per_track['monza-like']} (one per 10 waypoints), so the agent refuses to build with it, and even "
    "if the lengths matched, those offsets would push the car toward walls of a different track. "
    "When the race's track changes, the line must be solved again on the new track's centreline and widths "
    "(Unit 9), its grip sections measured again (Unit 8), and everything tuned on that line raced and checked again."
)
print(why_per_track)

assert len(monza_control) == 285 and len(monza_offsets) == len(track) == 2857
assert inside_width and pinned_ends and same_line and max_difference <= 1e-6
assert values_per_track == {"test-circuit": 133, "monza-like": 285}
assert 3.0 < largest_offset <= float(np.max(room)) + 1e-9

## Exercise 3 - The re-tune's record (`black-box-search`, `repeated-run-measurement`, `determinism-and-packaging`)

`P2` is the probe record of plan `roar-013`: the crash-aware coordinate search on `monza-like`, every setting it evaluated, in order.
Each evaluation raced the nominal car at race seeds 0 and 1 and the training cars 16, 18 and 8 (3 laps); its two **per-replicate sums** are the nominal race at seed r plus the three cars' times.
Each sum is written exactly as the search computed it (with `repr`, Unit 12's trap 1), so your comparisons are the search's.
For a setting that was not clean, the dict gives each failing race: car → (collisions, max line error in m).

**The search** (the lesson's §2, Unit 12's coordinate search): start from `unit-11.py`'s (0.54, 5.0, 4.0); for each knob in the order `MU_FRACTION`, `A_DEC`, `A_ACC`, look at the settings one grid step up and one down (steps 0.02, 0.5, 0.5, each rounded with `round(x, 2)`), take the one with the lower median sum (a setting that is not clean counts as infinitely slow; on a tie, up), and step to it if it **beats** the incumbent: clean, and `noise_aware_better(its sums, the incumbent's sums)`.
After a step, keep stepping the same way while each next setting beats the incumbent; then go on to the next knob.
Repeat the three knobs until a whole pass takes no step.
Every setting your replay needs is in `P2`.

`AT_A_ACC_5` gives four more settings at `A_ACC = 5.0`, raced on the same objective by the record (p3, p3b and p10).

Define:

- `replay(rule)`: the search on `P2` (and `AT_A_ACC_5`), where `rule(candidate_sums, incumbent_sums)` decides "beats" for two clean settings; it returns the list of settings taken, in order, starting with the start;
- `float_steps`: `replay(noise_aware_better)`, and `float_result`: its last setting (the search's literal output);
- `tie_step`: the step of `float_steps` whose two pairs of sums are **one tick apart** in their medians and **one tick** in their spreads (a tuple: the setting before, the setting after);
- `in_ticks`: a function that turns a list of sums into whole ticks (`round(t / DT)`), and `tick_rule(a, b)`: `noise_aware_better(in_ticks(a), in_ticks(b))`;
- `tick_steps`: `replay(tick_rule)`, and `incumbent`: its last setting;
- `edges`: a dict from each knob at an edge to the first value of it that is not clean next to `incumbent` (use `AT_A_ACC_5`), and `no_edge`: the knobs with no edge found;
- `not_clean_reasons`: a dict from each setting of `P2` that is not clean to a list of strings, one per failing race, `"collision"` when it had a collision and `"line error"` when it finished with none but more than 2 m from its line;
- `tie_explained`: three sentences: why does the float comparison say "beats" at `tie_step`, why is it not a gain, and what habit prevents it?

**Prediction.** The lesson's table says the search took `MU_FRACTION` 0.54 → 0.66 (0.68 is not clean), never moved `A_DEC` (5.5 is not clean, 4.5 is slower) and took `A_ACC` 4.0 → 4.5 → 5.0 → 5.5.
The last step is the float tie: 5.0's sums [1861.95, 1861.90] against 5.5's [1861.90, 1861.85], a gain of one tick against a spread of one tick.
In whole ticks that step is refused, so the tick replay should stop at (0.66, 5.0, 5.0), with edges at `MU_FRACTION` 0.68 and `A_DEC` 5.5 and no edge for `A_ACC`.

In [ ]:
# The probe record of plan roar-013 (p2): (MU_FRACTION, A_DEC, A_ACC) -> ((the two per-replicate sums in s, exactly
# as computed), {car: (collisions, max line error in m)} for each race that was not clean), in the order evaluated.
P2 = {
    (0.54, 5.0, 4.0): ((1929.9, 1929.8500000000001), {}),
    (0.56, 5.0, 4.0): ((1916.3000000000002, 1916.3000000000002), {}),
    (0.52, 5.0, 4.0): ((1943.9, 1943.75), {}),
    (0.58, 5.0, 4.0): ((1905.7, 1905.65), {}),
    (0.6, 5.0, 4.0): ((1895.5500000000002, 1895.5000000000002), {}),
    (0.62, 5.0, 4.0): ((1886.0, 1885.95), {}),
    (0.64, 5.0, 4.0): ((1876.8500000000001, 1876.8000000000002), {}),
    (0.66, 5.0, 4.0): ((1868.4499999999998, 1868.3999999999999), {}),
    (0.68, 5.0, 4.0): ((1959.6000000000001, 1959.5500000000002), {16: (0, 2.012), 8: (1, 4.510)}),
    (0.66, 5.5, 4.0): ((2130.6000000000004, 2130.6000000000004), {16: (0, 2.794), 18: (0, 2.805), 8: (2, 3.150)}),
    (0.66, 4.5, 4.0): ((1875.7500000000002, 1875.7500000000002), {}),
    (0.66, 5.0, 4.5): ((1862.5500000000002, 1862.6000000000001), {}),
    (0.66, 5.0, 3.5): ((1880.75, 1880.7), {}),
    (0.66, 5.0, 5.0): ((1861.95, 1861.9), {}),
    (0.66, 5.0, 5.5): ((1861.9, 1861.8500000000001), {}),
    (0.66, 5.0, 6.0): ((1862.0500000000002, 1861.95), {}),
    (0.68, 5.0, 5.5): ((2491.6, 2491.45), {16: (2, 6.846), 8: (2, 7.717)}),
    (0.64, 5.0, 5.5): ((1870.3000000000002, 1870.3000000000002), {}),
    (0.66, 5.5, 5.5): ((1856.15, 1856.0500000000002), {16: (0, 2.815), 18: (0, 2.804), 8: (0, 2.780)}),
    (0.66, 4.5, 5.5): ((1869.5000000000002, 1869.5000000000002), {}),
}
# The same objective at A_ACC 5.0, from the record's later probes (p3, p3b, p10), in the same form.
AT_A_ACC_5 = {
    (0.68, 5.0, 5.0): ((2532.9, 2532.8), {16: (2, 4.751), 18: (0, 2.013), 8: (2, 2.951)}),
    (0.64, 5.0, 5.0): ((1870.3000000000002, 1870.3500000000001), {}),
    (0.66, 5.5, 5.0): ((2515.4500000000003, 2515.3500000000004), {16: (2, 9.576), 18: (0, 2.795), 8: (2, 2.815)}),
    (0.66, 4.5, 5.0): ((1869.4, 1869.45), {}),
}
KNOBS = ("MU_FRACTION", "A_DEC", "A_ACC")
STEPS = (0.02, 0.5, 0.5)
START = (0.54, 5.0, 4.0)
TABLE = {}  # every setting the replay can look up: P2 and the four later ones at A_ACC 5.0
for setting, record in P2.items():
    TABLE[setting] = record
for setting, record in AT_A_ACC_5.items():
    TABLE[setting] = record


def is_clean(setting):
    return len(TABLE[setting][1]) == 0


def median_sum(setting):
    # a setting that is not clean counts as infinitely slow
    return float(np.median(TABLE[setting][0])) if is_clean(setting) else math.inf


def moved(setting, knob, direction):
    # one grid step of `knob`, up (direction +1) or down (-1), rounded to the grid
    values = list(setting)
    values[knob] = round(values[knob] + direction * STEPS[knob], 2)
    return (values[0], values[1], values[2])


def beats(candidate, current, rule):
    return is_clean(candidate) and rule(list(TABLE[candidate][0]), list(TABLE[current][0]))


def replay(rule):
    current = START
    taken = [START]
    stepped = True
    while stepped:  # one pass over the three knobs, until a pass takes no step
        stepped = False
        for knob in range(3):
            up, down = moved(current, knob, 1), moved(current, knob, -1)
            direction = 1 if median_sum(up) <= median_sum(down) else -1  # on a tie, up
            candidate = up if direction == 1 else down
            while beats(candidate, current, rule):
                current = candidate
                taken.append(current)
                stepped = True
                candidate = moved(current, knob, direction)
    return taken


def in_ticks(times):
    return [round(t / DT) for t in times]


def tick_rule(a, b):
    return noise_aware_better(in_ticks(a), in_ticks(b))


float_steps = replay(noise_aware_better)
float_result = float_steps[-1]
ties = []
for before, after in zip(float_steps[:-1], float_steps[1:]):
    a, b = in_ticks(TABLE[before][0]), in_ticks(TABLE[after][0])
    gap = float(np.median(a) - np.median(b))
    spread = max(max(a) - min(a), max(b) - min(b))
    if gap == 1.0 and spread == 1:
        ties.append((before, after))
tie_step = ties[0]
tick_steps = replay(tick_rule)
incumbent = tick_steps[-1]

edges, no_edge = {}, []
for knob, name in enumerate(KNOBS):
    setting = moved(incumbent, knob, 1)  # walk up from the incumbent while the record has the setting and it is clean
    while setting in TABLE and is_clean(setting):
        setting = moved(setting, knob, 1)
    if setting in TABLE:
        edges[name] = setting[knob]
    else:
        no_edge.append(name)

not_clean_reasons = {}
for setting, (sums, bad) in P2.items():
    if len(bad) > 0:
        not_clean_reasons[setting] = ["collision" if collisions > 0 else "line error"
                                      for collisions, error in bad.values()]

before_sums, after_sums = TABLE[tie_step[0]][0], TABLE[tie_step[1]][0]
float_gain = float(np.median(before_sums) - np.median(after_sums))
float_spread = max(max(before_sums) - min(before_sums), max(after_sums) - min(after_sums))
print("the search, compared as floats:", float_steps)
print("the search, compared in ticks: ", tick_steps)
print(f"the tie: {tie_step[0]} -> {tie_step[1]}: sums {before_sums} -> {after_sums}")
print(f"  as floats the gain {float_gain!r} s against the spread {float_spread!r} s: "
      f"beats {noise_aware_better(list(after_sums), list(before_sums))}")
print(f"  in ticks {in_ticks(before_sums)} -> {in_ticks(after_sums)}: beats {tick_rule(after_sums, before_sums)}")
print("incumbent:", incumbent, "| edges:", edges, "| no edge:", no_edge)
for setting, reasons in not_clean_reasons.items():
    print("not clean:", setting, reasons)
tie_explained = (
    f"The two pairs {list(before_sums)} and {list(after_sums)} are each one tick apart, but the sums are floats: "
    f"the medians' gap computes as {float_gain!r} s and the larger spread as {float_spread!r} s, so 'the gain is "
    "more than the spread' is true by about 2e-13 s and Unit 1's rule says 5.5 beats 5.0. "
    "It is not a gain: every race time is a whole number of 0.05 s ticks, so the gain is exactly one tick and the "
    "spread exactly one tick, and a gain that is no bigger than the noise is a tie (in ticks the rule says no, and "
    f"the search stops at {incumbent}). "
    "The habit is Unit 12's trap 3: compare race times, and sums of them, in whole ticks (round(t / DT)) or with a "
    "half-tick tolerance, never with a bare > or == on computed floats."
)
print(tie_explained)

TAKEN = [(0.54, 5.0, 4.0), (0.56, 5.0, 4.0), (0.58, 5.0, 4.0), (0.6, 5.0, 4.0), (0.62, 5.0, 4.0), (0.64, 5.0, 4.0),
         (0.66, 5.0, 4.0), (0.66, 5.0, 4.5), (0.66, 5.0, 5.0), (0.66, 5.0, 5.5)]  # the lesson's record of the steps
assert float_steps == TAKEN and float_result == (0.66, 5.0, 5.5)
assert tie_step == ((0.66, 5.0, 5.0), (0.66, 5.0, 5.5))
assert in_ticks(before_sums) == [37239, 37238] and in_ticks(after_sums) == [37238, 37237]
assert float_gain > 0.05 > float_spread
assert tick_steps == TAKEN[:-1] and incumbent == (0.66, 5.0, 5.0)
assert edges == {"MU_FRACTION": 0.68, "A_DEC": 5.5} and no_edge == ["A_ACC"]
assert not_clean_reasons == {
    (0.68, 5.0, 4.0): ["line error", "collision"],
    (0.66, 5.5, 4.0): ["line error", "line error", "collision"],
    (0.68, 5.0, 5.5): ["collision", "collision"],
    (0.66, 5.5, 5.5): ["line error", "line error", "line error"],
}

## Exercise 4 - The margin, on held-out cars (`perturbation-testing`, `held-out-track-validation`)

Unit 12 kept **tracks** apart: training, validation and test.
On race day the track is known and the **car** is not, so the same idea carries from tracks to cars: training cars the objective races, validation cars that judge a setting already found (and are spent once they change a choice), and test cars raced once at the end.

From Exercise 3's `incumbent` and `edges`, apply Unit 11's margin rule (ship one grid step inside the edge) two ways:

- **M**: the margin on Unit 11's knob, `MU_FRACTION`, only;
- **M2**: the margin on **every knob at its edge** (a knob with no edge keeps its value).

Race both on held-out car 41 (`race(seed=41 % 5, k=41, MU_FRACTION=..., A_DEC=..., A_ACC=...)`): two races, about 20 s.
`HELD_OUT_CLEAN` is the record's count of clean races on the 24 held-out cars k = 24 to 47 (p3, p3b, p3c), and `FRESH` M2's on two more sets.

Define:

- `margin_one_knob`, `margin_every_edge`: the two settings as `(MU_FRACTION, A_DEC, A_ACC)` tuples;
- `car41`: a dict `{"M": result, "M2": result}`, and `car41_clean`, `car41_collisions`: the same with `clean(result)` and the collision count;
- `n_over_n`: the sorted list of the names in `HELD_OUT_CLEAN` that are clean on all 24 held-out cars;
- `which_rule`: `"M"` or `"M2"`, the reading of the rule that is N/N on the held-out cars;
- `sets`: a dict from `"training"`, `"validation"` and `"test"` to the car numbers of each set in this unit, each a `range` (the lesson's §2);
- `spent`: two sentences: why are the cars 24 to 47 a **validation** set now and not a test, and why were cars 48 to 71 not kept as the test either?
- `tracks_to_cars`: two sentences: what plays the role of Unit 12's test **track** here, and what would racing it again (for example to try another setting) do to it?

**Prediction.** From Exercise 3, the edges are `MU_FRACTION` (0.68) and `A_DEC` (5.5), so M is (0.64, 5.0, 5.0) and M2 is (0.64, 4.5, 5.0), `unit-13.py` as shipped.
The lesson raced both on car 41: M collides once and its race costs about 686 s; M2 is clean in 467.40 s.
Only M2 is 24/24 in the record, so the every-edge reading is the one that holds.

In [ ]:
# The probe record of plan roar-013 (p3, p3b, p3c): clean races of 24 on the held-out cars k = 24-47, 3 laps.
HELD_OUT_CLEAN = {
    "E (0.66, 5.0, 5.0)": 22, "E' (0.66, 5.0, 5.5)": 22, "M (0.64, 5.0, 5.0)": 23, "M2 (0.64, 4.5, 5.0)": 24,
    "P (0.66, 4.5, 5.0)": 24, "unit-11.py": 24, "unit-12.py": 24, "unit-07.py": 12,
}
# M2 after the choice (p8: cars 48-71, later raced again in p9; p11: cars 72-95, raced once): (clean of 24, median s)
FRESH = {"cars 48-71": (24, 469.65), "cars 72-95": (24, 468.65)}

def with_margin(knobs):
    # Unit 11's rule on each named knob: one grid step inside its edge (the edge is above the incumbent here)
    values = list(incumbent)
    for knob, name in enumerate(KNOBS):
        if name in knobs:
            direction = 1 if edges[name] > incumbent[knob] else -1
            values[knob] = round(incumbent[knob] - direction * STEPS[knob], 2)
    return (values[0], values[1], values[2])


margin_one_knob = with_margin(["MU_FRACTION"])
margin_every_edge = with_margin(list(edges))
car41 = {}
for name, setting in (("M", margin_one_knob), ("M2", margin_every_edge)):
    car41[name] = race(seed=41 % 5, k=41, MU_FRACTION=setting[0], A_DEC=setting[1], A_ACC=setting[2])
car41_clean = {name: clean(result) for name, result in car41.items()}
car41_collisions = {name: result.collisions for name, result in car41.items()}
n_over_n = sorted([name for name, n_clean in HELD_OUT_CLEAN.items() if n_clean == 24])
which_rule = "M2" if HELD_OUT_CLEAN["M2 (0.64, 4.5, 5.0)"] == 24 else "M"
sets = {"training": range(0, 24), "validation": range(24, 72), "test": range(72, 96)}

for name, result in car41.items():
    print(f"{name} {(margin_one_knob if name == 'M' else margin_every_edge)}, car 41: {result.total_time:.2f} s, "
          f"{result.collisions} collisions, {line_error(result):.2f} m from its line, clean {car41_clean[name]}")
print("N/N on the held-out cars:", n_over_n)
print("sets:", {name: (cars[0], cars[-1]) for name, cars in sets.items()})
spent = (
    f"The held-out cars 24 to 47 chose a setting: M2's 24/24 against M's {HELD_OUT_CLEAN['M (0.64, 5.0, 5.0)']}/24 "
    "decided which reading of the rule ships, and their table then suggested P, so a setting picked by looking at "
    "them is now fitted to them as Unit 12's validation track was, and their verdict on it is no longer an "
    "unseen-car verdict. "
    f"Cars 48 to 71 were set aside as a test and raced M2 once ({FRESH['cars 48-71'][0]}/24 clean), but they were "
    "raced again under other timing settings and informed the timing decision, so they took part in a choice too; "
    "only the fresh cars 72 to 95, picked after the file was fixed and raced once, are still a test."
)
tracks_to_cars = (
    f"Unit 12's test track is played here by the test cars 72 to 95: set aside before anything was chosen and raced "
    f"once with the shipped file ({FRESH['cars 72-95'][0]}/24 clean, median {FRESH['cars 72-95'][1]:.2f} s). "
    "Racing them again to try another setting would let their results choose, which spends them: they would become "
    "validation cars, and the book would have no unseen car left to say how the shipped file does on a car it has "
    "never met."
)
print(spent)
print(tracks_to_cars)

assert margin_one_knob == (0.64, 5.0, 5.0) and margin_every_edge == (0.64, 4.5, 5.0)
assert car41_clean == {"M": False, "M2": True} and car41_collisions == {"M": 1, "M2": 0}  # U13.3, p3
assert abs(car41["M"].total_time - 685.55) <= TICK and abs(car41["M2"].total_time - 467.40) <= TICK  # p3
assert n_over_n == ["M2 (0.64, 4.5, 5.0)", "P (0.66, 4.5, 5.0)", "unit-11.py", "unit-12.py"]
assert which_rule == "M2"
assert sets == {"training": range(24), "validation": range(24, 72), "test": range(72, 96)}

## Exercise 5 - MPC or pursuit (`ltv-mpc`, `pure-pursuit`)

Unit 10's question at the race-day setting.
Race `unit-13.py` as shipped and the same file with `STEERING = "pursuit"` on held-out car 34 (race seed 4): two races, about 15 s.
`P4` is the record's nominal table (race seeds 0 to 4), and `HELD_OUT` the clean counts on the 24 held-out cars.
Decide by Unit 10's order: **time first** (Unit 1's rule, either way round), **then margin** (the line error and the held-out clean counts).

Define:

- `car34`: a dict `{"mpc": result, "pursuit": result}`; `car34_clean` and `car34_error`: the same with `clean(result)` and `line_error(result)`;
- `faster`: `"mpc"`, `"pursuit"` or `"neither"`, by Unit 1's rule on `P4`'s times;
- `margin_winner`: the steering with the smaller nominal max line error **and** more clean held-out cars (`None` if neither has both);
- `choice`: the steering you ship, by Unit 10's order;
- `cpu_ratio`: MPC's computer time per race over pursuit's (`P4_CPU`);
- `why_mpc`: two sentences: what does MPC buy here that pursuit does not, and why is its cost (`cpu_ratio`) acceptable in the official race?

**Prediction.** `P4`'s two lists are the same race to a tick or two (medians 468.55 and 468.60 s with spreads of 0.10 s), so Unit 1's rule should say neither is faster.
On margin MPC wins both ways (1.01 m against 1.66 m, 24 against 13 held-out cars), and the lesson showed pursuit colliding once on car 34, where MPC should be clean.
MPC costs about 2.7 times the computer time.

In [ ]:
# The probe record of plan roar-013 (p4, p3c): the race-day setting, nominal car, race seeds 0-4, 3 laps.
P4 = {"mpc": [468.55, 468.55, 468.55, 468.65, 468.60], "pursuit": [468.55, 468.55, 468.60, 468.65, 468.60]}  # s
P4_MAX_ERROR = {"mpc": 1.009, "pursuit": 1.660}  # m: the largest line error over the five races
P4_CPU = {"mpc": 9.8, "pursuit": 3.6}  # s of computer time per race
HELD_OUT = {"mpc": 24, "pursuit": 13}  # clean of the 24 held-out cars k = 24-47

car34 = {"mpc": race(seed=34 % 5, k=34), "pursuit": race(seed=34 % 5, k=34, STEERING="pursuit")}
car34_clean = {name: clean(result) for name, result in car34.items()}
car34_error = {name: line_error(result) for name, result in car34.items()}
if noise_aware_better(P4["mpc"], P4["pursuit"]):
    faster = "mpc"
elif noise_aware_better(P4["pursuit"], P4["mpc"]):
    faster = "pursuit"
else:
    faster = "neither"
margin_winner = None
for name, other in (("mpc", "pursuit"), ("pursuit", "mpc")):
    if P4_MAX_ERROR[name] < P4_MAX_ERROR[other] and HELD_OUT[name] > HELD_OUT[other]:
        margin_winner = name
choice = faster if faster != "neither" else margin_winner
cpu_ratio = P4_CPU["mpc"] / P4_CPU["pursuit"]

for name, result in car34.items():
    print(f"{name:8s} car 34: {result.total_time:.2f} s, {result.collisions} collisions, "
          f"{car34_error[name]:.2f} m from its line, clean {car34_clean[name]}")
for name, times in P4.items():
    print(f"{name:8s} nominal median {np.median(times):.2f} s, spread {max(times) - min(times):.2f} s, "
          f"max {P4_MAX_ERROR[name]:.2f} m, held-out {HELD_OUT[name]}/24 clean")
print(f"faster: {faster}; margin: {margin_winner}; ship: {choice}; MPC costs {cpu_ratio:.2f} x pursuit's CPU")
why_mpc = (
    "MPC buys margin, not speed: on the nominal car the two are a tie by Unit 1's rule, but MPC stays closer to its "
    f"line ({P4_MAX_ERROR['mpc']:.2f} m against {P4_MAX_ERROR['pursuit']:.2f} m) and is clean on "
    f"{HELD_OUT['mpc']}/24 held-out cars against pursuit's {HELD_OUT['pursuit']}/24 (on car 34 pursuit runs "
    f"{car34_error['pursuit']:.2f} m off and collides once, costing "
    f"{car34['pursuit'].total_time - car34['mpc'].total_time:.0f} s). "
    f"Its cost, {cpu_ratio:.1f} times the computer time per race, does not count in the official race: what counts is "
    "each step() inside the tick budget, and MPC's step p99 of about 1.0 ms is five times inside the 5 ms budget, "
    "checked by check_submission on the race machine."
)
print(why_mpc)

assert car34_clean == {"mpc": True, "pursuit": False} and car34["pursuit"].collisions == 1  # U13.4
assert abs(car34["mpc"].total_time - 468.75) <= TICK  # p3c: M2 on car 34 (CAR_TIMES in Exercise 6)
assert car34_error["mpc"] <= CLEAN_ERROR < car34_error["pursuit"]
assert faster == "neither" and margin_winner == "mpc" and choice == "mpc"
assert abs(cpu_ratio - 9.8 / 3.6) <= 1e-12

## Exercise 6 - What a crash costs (`respawn-economics`)

Unit 11's **expected race time**: the mean race time over a family of cars, every race counted at its time (a crash costs what the respawn costs), the setting with the shortest wins.
`CAR_TIMES` gives the record's 24 race times on the held-out cars k = 24 to 47 (3 laps, race seed k mod 5) for the edge E (0.66, 5.0, 5.0), M (0.64, 5.0, 5.0), M2 (0.64, 4.5, 5.0) and `unit-07.py`; **a race cut at 900 s counts at 900 s** (the stated crash cost of a race that never finished).
`NOT_CLEAN` names the cars whose race was not clean.
No races.

Define:

- `expected`: a dict from each name to its expected race time (s); `expected_pick`: the name with the shortest;
- `clean_mean_e`: the mean of E's clean races, `crash_mean_e`: the mean of E's races that were not clean, and `saving`: `expected["M2"] - clean_mean_e`, what E saves in a race that does not crash (s);
- `observed_rate_e`: the fraction of E's 24 races that were not clean;
- `break_even_rate`: the crash rate p at which E's expected time, (1 − p) · `clean_mean_e` + p · `crash_mean_e`, equals M2's;
- `fig`, `ax`: the 24 times of each setting against the car number, the not-clean races marked, with a legend, axis labels and a title;
- `checkpoint_b`: three sentences: in Checkpoint B, expected time picked a fraction (0.54) past the family's edge (0.50), and that pick beat the margin rule's 0.48 while fewer than 4.5% of races crashed.
  Why does expected time pick the margin setting here instead, and what does the comparison of `observed_rate_e` with `break_even_rate` say?
  What does either rule assume about how often a car crashes on race day?

**Prediction.** The lesson quotes the record: E 494.11 s, M 476.24 s, M2 468.98 s, `unit-07.py` 707.62 s, so expected time picks M2.
E saves about 3.9 s a clean race, and its two crashes (737 s and 889 s) cost about 345 s each against a clean race, so the break-even crash rate should be about 1%, well below E's observed 2 in 24 (8.3%).

In [ ]:
# The probe record of plan roar-013 (p3, p3c): 3-lap race times (s) on held-out cars k = 24-47, in order of k.
CAR_TIMES = {
    "E": [737.25, 462.90, 464.95, 465.15, 464.20, 467.20, 465.70, 466.65, 466.95, 463.25, 464.75, 466.40,
          464.10, 463.75, 467.20, 465.75, 465.90, 889.40, 464.80, 464.80, 464.90, 463.10, 463.50, 466.20],
    "M": [466.75, 464.95, 467.05, 467.25, 466.25, 469.20, 467.75, 468.70, 469.05, 465.30, 466.80, 468.40,
          466.25, 465.80, 469.35, 467.75, 467.90, 685.55, 466.85, 466.75, 467.00, 465.10, 465.65, 468.35],
    "M2": [468.60, 466.85, 468.95, 469.15, 468.20, 471.10, 469.75, 470.70, 470.90, 467.25, 468.75, 470.30,
           468.15, 467.75, 471.20, 469.55, 469.85, 467.40, 468.75, 468.75, 468.90, 467.00, 467.55, 470.20],
    "unit-07.py": [900.00, 514.55, 515.35, 515.55, 514.85, 900.00, 515.55, 515.75, 900.00, 900.00, 900.00, 900.00,
                   900.00, 514.90, 516.35, 900.00, 515.50, 900.00, 514.90, 900.00, 515.35, 514.40, 900.00, 900.00],
}
CARS = list(range(24, 48))
NOT_CLEAN = {"E": [24, 41], "M": [41], "M2": [], "unit-07.py": [24, 29, 32, 33, 34, 35, 36, 39, 41, 43, 46, 47]}

expected = {name: float(np.mean(times)) for name, times in CAR_TIMES.items()}
expected_pick = sorted(expected, key=lambda name: expected[name])[0]
e_clean = [t for car, t in zip(CARS, CAR_TIMES["E"]) if car not in NOT_CLEAN["E"]]
e_crash = [t for car, t in zip(CARS, CAR_TIMES["E"]) if car in NOT_CLEAN["E"]]
clean_mean_e, crash_mean_e = float(np.mean(e_clean)), float(np.mean(e_crash))
saving = expected["M2"] - clean_mean_e
observed_rate_e = len(NOT_CLEAN["E"]) / len(CARS)
break_even_rate = (expected["M2"] - clean_mean_e) / (crash_mean_e - clean_mean_e)

fig, ax = plt.subplots(figsize=(10, 4))
for name, times in CAR_TIMES.items():
    ax.plot(CARS, times, "o-", markersize=3, linewidth=0.7, label=name)
bad_cars = [car for name in CAR_TIMES for car in NOT_CLEAN[name]]
bad_times = [CAR_TIMES[name][car - 24] for name in CAR_TIMES for car in NOT_CLEAN[name]]
ax.plot(bad_cars, bad_times, "kx", markersize=9, label="not clean")
ax.set_xlabel("held-out car k (race seed k mod 5)")
ax.set_ylabel("3-lap race time (s; cut at 900)")
ax.set_title("monza-like, held-out cars 24-47: what a crash costs")
ax.legend()
plt.show()

for name in CAR_TIMES:
    print(f"{name:11s} expected {expected[name]:7.2f} s, not clean on {NOT_CLEAN[name]}")
print(f"pick: {expected_pick}; E clean mean {clean_mean_e:.2f} s, crash mean {crash_mean_e:.2f} s, "
      f"saving {saving:.2f} s a clean race")
print(f"E's crash rate {observed_rate_e:.4f} against the break-even rate {break_even_rate:.4f}")
checkpoint_b = (
    f"Here a crash costs E about {crash_mean_e - clean_mean_e:.0f} s against the {saving:.2f} s it saves in a clean race, "
    f"so its two crashes in 24 races lift its expected time to {expected['E']:.2f} s, above M2's "
    f"{expected['M2']:.2f} s, and expected time picks the margin setting, the same answer as the margin rule. "
    f"E's observed crash rate, {observed_rate_e:.3f} (2 of 24), is about {observed_rate_e / break_even_rate:.0f} times "
    f"the break-even rate {break_even_rate:.4f} (about one race in {1 / break_even_rate:.0f}), so E would have to crash "
    "far less often to pay; in Checkpoint B the edge crashed rarely enough to stay under its break-even rate, and here "
    "it does not. "
    "Expected time assumes race day's crash rate is the one measured on the family (and that the family is like "
    "the real car), while the margin rule assumes nothing about the rate and buys insurance against a real car "
    "that crashes more often than any sample showed; with 24 cars a rate of 1 in 100 cannot even be told from 0."
)
print(checkpoint_b)

assert {name: round(value, 2) for name, value in expected.items()} == {
    "E": 494.11, "M": 476.24, "M2": 468.98, "unit-07.py": 707.62}  # the record
assert expected_pick == "M2"
assert abs(crash_mean_e - 813.325) <= 1e-9 and abs(saving - 3.89) <= 0.005
assert abs(observed_rate_e - 2 / 24) <= 1e-12
assert abs(break_even_rate - saving / (crash_mean_e - clean_mean_e)) <= 1e-12 and 0.01 < break_even_rate < 0.015
assert break_even_rate < observed_rate_e

## Exercise 7 - The timing, on and off (`tick-budget`)

`unit-13.py` ships with the timing on (`MPC_BUDGET_MS = 5.0`) and `MPC_MIN_HORIZON = MPC_HORIZON = 40`.
Race it twice on the nominal car, race seed 0, **1 lap** (`laps=1`, about 7 s in all): once exactly as shipped (`run_race(UNIT_13, TRACK, car=carla_like(noise=True), seed=0, laps=1, max_seconds=CAP)`: the timing on), and once with the timing off (`race(laps=1)`).
Your step times depend on the machine and its load: **print** them and read them; do not compare them with a fixed number.

Then, without a race, build two agents (as in Exercise 1) from `variant(UNIT_13, MPC_BUDGET_MS=0.0)` and `variant(UNIT_13, MPC_BUDGET_MS=0.0, MPC_MIN_HORIZON=10)`, and call each one's `mpc_command(0.0, 15.0)` three times: every call is over a budget of 0 ms.
`P9` is the record's bracket of what Unit 10's halving policy would have raced.

Define:

- `timed`, `untimed`: the two races; `step_ms`: a dict `{"on": (p50, p99), "off": (p50, p99)}` of each race's `step_ms` percentiles (print it);
- `same_time`: `True` when the two race times are equal; `same_telemetry`: `same_race(timed, untimed)`;
- `horizons`: a dict `{40: [...], 10: [...]}` from each `MPC_MIN_HORIZON` to the agent's `horizon` after each of its three calls, and `counted`: a dict from each to its `over_budget` after the three calls;
- `why_same`: two sentences: why are the timing-on and timing-off races the same race with this file, when Unit 12's four timing-on races of `unit-11.py` were four different races?
- `why_counter`: three sentences: from `P9`, was each regime of the halving policy clean, what can no finite set of races cover, and why does the race-day file turn the policy into a counter (and what must then be checked before the race)?

**Prediction.** With `MPC_MIN_HORIZON = MPC_HORIZON = 40` a slow call can only add to `over_budget`; nothing the car does reads the clock, so the timing-on and timing-off races should be the same race, row for row, and the same time.
The step times should be about a millisecond or less, but they are the machine's, not the race's.
With a budget of 0 ms, the race-day file keeps the horizon at 40 for all three calls, and Unit 10's policy (`MPC_MIN_HORIZON = 10`) halves it to 20 and then 10; both count three calls over budget.

In [ ]:
# The probe record of plan roar-013 (p9): the race-day setting with the horizon fixed at 20, and with
# MPC_BUDGET_MS = 0.0 (every call over budget: the horizon 10 from the third tick), 3 laps, timing off.
# (nominal clean of 5, cars 24-47 clean of 24, cars 48-71 clean of 24, the largest line error in m)
P9 = {"horizon 20": (5, 24, 24, 1.382), "horizon 10 (budget 0.0)": (5, 24, 24, 1.579)}

timed = run_race(UNIT_13, TRACK, car=carla_like(noise=True), seed=0, laps=1, max_seconds=CAP)  # the timing on
untimed = race(laps=1)  # the timing off
step_ms = {name: (float(np.percentile(result.step_ms, 50)), float(np.percentile(result.step_ms, 99)))
           for name, result in (("on", timed), ("off", untimed))}
same_time = timed.total_time == untimed.total_time
same_telemetry = same_race(timed, untimed)

horizons, counted = {}, {}
for minimum in (40, 10):
    with shim_installed():
        Agent, _ = load_submission(variant(UNIT_13, MPC_BUDGET_MS=0.0, MPC_MIN_HORIZON=minimum))
    agent = Agent(track.waypoints, None)
    horizons[minimum] = []
    for call in range(3):
        agent.mpc_command(0.0, 15.0)
        horizons[minimum].append(agent.horizon)
    counted[minimum] = agent.over_budget

print(f"1 lap, seed 0: timing on {timed.total_time:.2f} s, timing off {untimed.total_time:.2f} s; "
      f"same time {same_time}, same telemetry {same_telemetry}")
print("step_ms (p50, p99) on this machine, now:", {name: (round(a, 3), round(b, 3)) for name, (a, b) in step_ms.items()})
print("horizons after each call:", horizons, "| over_budget:", counted)
why_same = (
    "In unit-13.py the clock can only add to over_budget: a slow mpc() call would halve the horizon, but "
    "MPC_MIN_HORIZON = MPC_HORIZON = 40 makes the halved horizon max(20, 40) = 40, so no command, plan or state "
    f"depends on the wall clock and the two races are the same race (here {timed.total_time:.2f} s, every telemetry "
    "row equal). "
    "unit-11.py's MPC_MIN_HORIZON is 10, so in Unit 12 a call that happened to be slow under load cut the horizon to "
    "20 or 10 at a different tick in each run, and from that tick on each race drove a different plan."
)
why_counter = (
    f"P9 says each regime is clean: the horizon fixed at 20 and the horizon at 10 from the third tick were both "
    f"{P9['horizon 20'][0]}/5 nominal, {P9['horizon 20'][1]}/24 on cars 24-47 and {P9['horizon 20'][2]}/24 on cars "
    f"48-71 (max line error {P9['horizon 20'][3]} m and {P9['horizon 10 (budget 0.0)'][3]} m). "
    "But a load spike can cut the horizon at any tick of a 9,000-tick race, on any car, and a mid-race switch from "
    "40 to 20 or 10 is a different race at every tick, so no finite set of races covers every switch. "
    "Making the policy a counter means the shipped file always races the behaviour the gates raced; the price is "
    "that it no longer protects itself on a slow machine, so before the race check_submission must be run on the race "
    "machine, idle, and its step p99 must be inside the 5 ms budget."
)
print(why_same)
print(why_counter)

assert same_time and same_telemetry and timed.finished and timed.collisions == 0
assert abs(untimed.total_time - 159.15) <= TICK
assert clean(timed) and clean(untimed)
assert horizons == {40: [40, 40, 40], 10: [20, 10, 10]} and counted == {40: 3, 10: 3}  # U13.0b

## Exercise 8 - The packaging checklist (`determinism-and-packaging`)

Check the file you would submit as a reviewer would, by reading it.
`module_constants` (the setup) gives every module constant's value as text, without its comment; `file_parts` gives the header comment lines and the rest of the file without its constants and the line's values.

Define:

- `texts`: `module_constants(UNIT_13)`; `pinned_ok`: a dict from each name in `PINNED` to `True` when the file's text for it is exactly the pinned text;
- `round_trips`: a dict from each **float** constant of `PINNED` (`MU_FRACTION`, `A_DEC`, `A_ACC`, `MPC_BUDGET_MS`) to `True` when `float(text) == the value the module holds` (`thirteen.MU_FRACTION` and so on);
- `imports`: the file's import lines (from `file_parts`'s second part: lines starting with `"import "`, or with `"from "` and containing `" import "`), and `allowed_imports`: `True` when each imports only `__future__`, `math`, `time`, `typing`, `numpy` or `roar_py_interface`;
- `header_says`: a dict from each of the six changed constants' names to `True` when it is mentioned in the header;
- `default_report`: `check_submission(UNIT_13)` (its default track, `test-circuit`; cheap: it fails as the agent is built), and `refused_for_the_line`: `True` when the report failed and its text names both 285 and 133;
- `checklist`: a list of at least eight strings, each one thing to check before you submit (the lesson's §6: the file, Python, the values, the timing, the check and where to run it, the versions and the hash, the season's files and wording);
- `why_own_track`: two sentences: why must a race-day file be checked on its own track, and why is the **timed** check run on the race machine, idle?

**Prediction.** The file should carry every pinned text exactly, each float should round-trip, and its imports are the lesson's six (`__future__`, `math`, `time`, `typing`, `numpy`, `roar_py_interface`).
The header names the six changed constants.
On `test-circuit` the check should fail as the agent is built: `LINE_CONTROL has 285 values, this track needs 133`.

In [ ]:
# The values the race-day file must carry, as text (plan roar-013, "the reference submission").
PINNED = {"MU_FRACTION": "0.64", "A_DEC": "4.5", "A_ACC": "5.0", "MPC_MIN_HORIZON": "40", "MPC_HORIZON": "40",
          "MPC_BUDGET_MS": "5.0", "STEERING": '"mpc"', "MU_G": "(5.38, 8.96, 9.12, 9.14, 9.7, 7.97)"}

texts = module_constants(UNIT_13)
pinned_ok = {name: texts[name] == text for name, text in PINNED.items()}
round_trips = {name: float(texts[name]) == getattr_value
               for name, getattr_value in (("MU_FRACTION", thirteen.MU_FRACTION), ("A_DEC", thirteen.A_DEC),
                                           ("A_ACC", thirteen.A_ACC), ("MPC_BUDGET_MS", thirteen.MPC_BUDGET_MS))}
header, body = file_parts(UNIT_13)
imports = [line[:-1] for line in body if line[:7] == "import " or (line[:5] == "from " and " import " in line)]
ALLOWED = ["__future__", "math", "time", "typing", "numpy", "roar_py_interface"]


def module_of(line):
    # the module an import line imports from: the word after "import " or "from ", up to a space, dot or comma
    rest = line[5:] if line[:5] == "from " else line[7:]
    ends = [i for i in range(len(rest)) if rest[i] in " .,"]
    return rest[:ends[0]] if ends else rest


allowed_imports = len(imports) > 0 and all(module_of(line) in ALLOWED for line in imports)
header_text = "".join(header)
CHANGED = ["LINE_CONTROL", "MU_G", "MU_FRACTION", "A_DEC", "A_ACC", "MPC_MIN_HORIZON"]
header_says = {name: name in header_text for name in CHANGED}
default_report = check_submission(UNIT_13)  # its default track, test-circuit
report_text = str(default_report)
refused_for_the_line = (not default_report.ok) and "285" in report_text and "133" in report_text
checklist = [
    "One file, submission.py, in the official shape: class RoarCompetitionSolution with async initialize and step.",
    "It imports only the standard library, numpy and roar_py_interface (here: " + ", ".join(ALLOWED) + ").",
    "It parses as Python 3.8, the official environment (check_submission parses it at Python 3.8).",
    "Every tuned value is on its grid and written with repr, and reads back as the value raced (round trips).",
    "The header comment says which constants changed, by which search, and why.",
    "The line (LINE_CONTROL) and the grip (MU_G) are the race track's, Monza's, not another track's.",
    "The timing is as shipped: MPC_BUDGET_MS = 5.0, with MPC_MIN_HORIZON = MPC_HORIZON so it only counts.",
    "check_submission passes on the race track (track='monza-like', or Monza on CARLA), on the race machine, "
    "idle, with step() p99 inside the 5 ms budget.",
    "Python, numpy and racekit versions written down, and the file's SHA-256 from the operating system's tool, "
    "so you can prove which file raced.",
    "The repository holds the unmodified competition_runner.py and infrastructure.py from the competition's fork.",
    "The season's page read fresh: instruction.txt or requirements.txt (the two sources disagree), the deadline, "
    "and what it said written down with the date.",
    "If the season changed the runner, the Monza map, CARLA or Python, the laps, line, grip and timing are re-checked.",
]
print("pinned texts:", pinned_ok)
print("round trips:", round_trips)
print("imports:", imports, "| allowed:", allowed_imports)
print("header names:", header_says)
print(default_report)
print("refused for the line:", refused_for_the_line)
for number, item in enumerate(checklist, start=1):
    print(f"{number:2d}. {item}")
why_own_track = (
    "The race-day file carries one track's line (285 values for monza-like's 2,857 waypoints) and that track's grip, "
    "so on any other track the agent refuses to build (here: 285 against test-circuit's 133), and even a check that "
    "passed elsewhere would not have raced the line, corners and timing of the race. "
    "The timed check measures step() on the machine that runs it, so it says nothing about a different or a busy "
    "machine; with the horizon policy turned into a counter the file no longer protects itself, so the p99 must be "
    "measured on the race machine itself, idle as it will be on race day."
)
print(why_own_track)

assert pinned_ok == {name: True for name in PINNED}
assert round_trips == {"MU_FRACTION": True, "A_DEC": True, "A_ACC": True, "MPC_BUDGET_MS": True}
assert imports == ["from __future__ import annotations", "import math", "import time",
                   "from typing import List, Optional, Sequence, Tuple", "import numpy as np", "import roar_py_interface"]
assert allowed_imports and header_says == {name: True for name in CHANGED}
assert refused_for_the_line and default_report.step_p99_ms is None
assert len(checklist) >= 8

## Exercise 9 - Challenge: a finding, or a fit? (`held-out-track-validation`, `perturbation-testing`)

P (0.66, 4.5, 5.0) is the margin on `A_DEC` only, the one knob whose next step failed on every low-grip car.
The record (p3b) found it clean on all 24 held-out cars and about 2 s faster than M2 there: median 466.73 s against 468.83 s.
P was raced **after** the held-out cars had already chosen M2, because the table showed a gap worth asking about.

Race P on held-out car 41 (race seed 1) and on the nominal car at race seed 0: two races, about 20 s.
Do **not** race cars 72 to 95 (the `race` helper refuses): they are the test cars, raced once, and nothing you decide here may depend on them.

Define:

- `p_car41`, `p_seed0`: the two results, and `p_clean`: a dict `{"car 41": ..., "seed 0": ...}` of `clean(result)`;
- `gain_seed0`: `unit-13.py`'s seed-0 time, 468.55 s (the milestone's pin), minus P's (s), and `gain_held_out`: 468.83 − 466.73 (s);
- `finding_or_fit`: three sentences: P was chosen by looking at the held-out cars' results; what does that do to their power to judge it, and is 2 s on the cars that suggested it evidence that P is faster on a car you have not met?
- `what_to_race`: two sentences: what would you need to race to tell, how would you fix it **before** racing, and which cars must it never be?

**Prediction.** P was 24/24 on the held-out cars, so on car 41 it should be clean, and on seed 0 it should be about 2 s faster than `unit-13.py`'s 468.55 s, like its held-out median.
But the 2 s was measured on the very cars that suggested P, so those cars cannot also judge it.

In [ ]:
P_SETTING = {"MU_FRACTION": 0.66, "A_DEC": 4.5, "A_ACC": 5.0}

p_car41 = race(seed=41 % 5, k=41, **P_SETTING)
p_seed0 = race(seed=0, **P_SETTING)
p_clean = {"car 41": clean(p_car41), "seed 0": clean(p_seed0)}
gain_seed0 = 468.55 - p_seed0.total_time
gain_held_out = 468.83 - 466.73

for label, result in (("P, car 41", p_car41), ("P, seed 0", p_seed0)):
    print(f"{label:10s} {result.total_time:.2f} s, {result.collisions} collisions, "
          f"{line_error(result):.2f} m from its line, clean {clean(result)}")
print(f"gain over unit-13.py at seed 0: {gain_seed0:.2f} s; on the held-out cars: {gain_held_out:.2f} s")
finding_or_fit = (
    "P was raced because the held-out table showed a gap, so the held-out cars have already been used to choose it: "
    "they are its training data now, and P's 24/24 and 2 s there are the score of a setting picked to look good on "
    "exactly those cars. "
    f"On them, and here (clean on car 41, {gain_seed0:.2f} s faster on the nominal car at seed 0), P is faster, but a "
    f"gain measured on the cars that suggested it is a fit until shown otherwise: {gain_held_out:.2f} s on cars 24-47 "
    "is not evidence that P is faster, or still clean, on a car you have not met. "
    "It is a hypothesis worth testing, not a finding; P takes the margin off MU_FRACTION, the knob whose edge "
    "(0.68) failed on low-grip cars, so its risk is exactly the kind a fresh car could expose."
)
what_to_race = (
    "Race P and M2 on a fresh set of perturbed cars that nothing has raced, chosen and written down before the "
    "first race, with the comparison fixed in advance (P must be clean on all of them and beat M2 by Unit 1's rule on "
    "their times, and otherwise M2 ships), and race them once. "
    "They must never be the test cars 72 to 95, which already judged the shipped file once: racing P on them would let "
    "them choose and spend the only unseen test the unit has, nor the training cars 0-23 or the validation cars 24-71 "
    "(and never the sealed k >= 1000)."
)
print(finding_or_fit)
print(what_to_race)

assert p_clean == {"car 41": True, "seed 0": True} and p_car41.collisions == 0 and p_seed0.collisions == 0
assert abs(p_car41.total_time - 465.40) <= TICK and abs(p_seed0.total_time - 466.45) <= TICK
assert abs(gain_held_out - 2.10) <= 1e-9 and abs(gain_seed0 - 2.10) <= TICK